In [1]:
# importing required libraries
import pandas as pd
import requests
from sqlalchemy import create_engine, text

In [2]:
import logging

logging.basicConfig(
    level=logging.INFO,                       # minimum severity to capture
    format="%(asctime)s | %(levelname)s | %(message)s",   # timestamp | severity | message
    handlers=[
        logging.FileHandler("etl_pipeline.log"),   # write to file
        logging.StreamHandler()                     # also show on console
    ]
)

logger = logging.getLogger(__name__)

Day 1: Extract

Part A:

In [3]:
# --- Source 1: CSV ---
file_path = "/home/vinaygautam/etl_learning/data/Netflix TV Shows and Movies.csv"

try:
    df_nf = pd.read_csv(file_path)
    logger.info(f"Extracted {df_nf.shape[0]} rows, {df_nf.shape[1]} columns\n")
    print(df_nf.dtypes, "\n")
    print(df_nf.head(5), "\n")
except FileNotFoundError:
    logger.error(f"Error: File not found at {file_path}")
except pd.errors.EmptyDataError:
    logger.error(f"Error: File is Empty")
except pd.errors.ParserError as e:
    logger.error(f"Error: Could not parse CSV - {e}")

2026-07-18 21:15:30,212 | INFO | Extracted 5283 rows, 11 columns



index                  int64
id                       str
title                    str
type                     str
description              str
release_year           int64
age_certification        str
runtime                int64
imdb_id                  str
imdb_score           float64
imdb_votes           float64
dtype: object 

   index        id                            title   type  \
0      0   tm84618                      Taxi Driver  MOVIE   
1      1  tm127384  Monty Python and the Holy Grail  MOVIE   
2      2   tm70993                    Life of Brian  MOVIE   
3      3  tm190788                     The Exorcist  MOVIE   
4      4   ts22164     Monty Python's Flying Circus   SHOW   

                                         description  release_year  \
0  A mentally unstable Vietnam War veteran works ...          1976   
1  King Arthur, accompanied by his squire, recrui...          1975   
2  Brian Cohen is an average young Jewish man, bu...          1979   
3  12-year-o

Part B:

In [ ]:
# --- Source 2: API ---
try:
    response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
    logger.info(f"API extraction successfull: (Status {response.status_code})")

    if response.status_code == 200:
        data = response.json()
        df_users = pd.json_normalize(data)
        logger.info(f"Extracted {df_users.shape[0]} rows, {df_users.shape[1]} columns\n")
        print(df_users.dtypes, "\n")
        print(df_users.head(5), "\n")
    else:
        logger.error(f"Something went wrong — status code {response.status_code}")
except requests.exceptions.ConnectionError:
    logger.error("Error: Could not connect — check your network or the API URL")
except requests.exceptions.Timeout:
    logger.error("Error: Request timed out")
except requests.exceptions.RequestException as e:
    logger.error(f"Error: Request failed — {e}")

2026-07-18 21:15:30,706 | INFO | API extraction successfull: (Status 200
2026-07-18 21:15:30,717 | INFO | Extracted 10 rows, 15 columns



id                     int64
name                     str
username                 str
email                    str
phone                    str
website                  str
address.street           str
address.suite            str
address.city             str
address.zipcode          str
address.geo.lat          str
address.geo.lng          str
company.name             str
company.catchPhrase      str
company.bs               str
dtype: object 

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    0

In [5]:
# --- Save raw staging layer ---
df_nf.to_parquet("/home/vinaygautam/etl_learning/data/raw/netflix_raw.parquet")
df_users.to_parquet("/home/vinaygautam/etl_learning/data/raw/users_raw.parquet")

In [6]:
# Verify raw save
netflix_check = pd.read_parquet("/home/vinaygautam/etl_learning/data/raw/netflix_raw.parquet")
users_check = pd.read_parquet("/home/vinaygautam/etl_learning/data/raw/users_raw.parquet")

logger.info(f"Netflix — original: {df_nf.shape}, read back: {netflix_check.shape}")
logger.info(f"Users API — original: {df_users.shape}, read back: {users_check.shape}")

2026-07-18 21:15:31,069 | INFO | Netflix — original: (5283, 11), read back: (5283, 11)
2026-07-18 21:15:31,072 | INFO | Users API — original: (10, 15), read back: (10, 15)


Day 2: Transform

--- Part A: Netflix ---

In [7]:
# Read the data from netflix raw file
df_nf_trfm = pd.read_parquet("/home/vinaygautam/etl_learning/data/raw/netflix_raw.parquet")

# Null check
print(df_nf_trfm.isnull().sum())

# Duplicate check
print(df_nf_trfm.duplicated().value_counts())

# Handle nulls
df_nf_trfm.loc[df_nf_trfm['description'].isnull(), 'description'] = "Description not available"
df_nf_trfm.loc[df_nf_trfm['age_certification'].isnull(), 'age_certification'] = "Certification not available"

# imdb_votes left as NaN (numeric measurement — no fabrication)

# Drop redundant index column
df_nf_trfm = df_nf_trfm.drop('index', axis=1)

print(df_nf_trfm.shape)
print(df_nf_trfm.isnull().sum())

index                   0
id                      0
title                   0
type                    0
description             5
release_year            0
age_certification    2285
runtime                 0
imdb_id                 0
imdb_score              0
imdb_votes             16
dtype: int64
False    5283
Name: count, dtype: int64
(5283, 10)
id                    0
title                 0
type                  0
description           0
release_year          0
age_certification     0
runtime               0
imdb_id               0
imdb_score            0
imdb_votes           16
dtype: int64


--- Part B: Users ---

In [8]:
# Read the data from users raw file
df_users_trfm = pd.read_parquet("/home/vinaygautam/etl_learning/data/raw/users_raw.parquet")

# Null + duplicate check
print(df_users_trfm.isnull().sum())
print(df_users_trfm.duplicated().value_counts())

# Drop unneeded columns + rename to flat schema
df_users_trfm = df_users_trfm.drop(
    {'username', 'address.geo.lat', 'address.geo.lng', 'company.catchPhrase', 'company.bs'},
    axis=1
)
df_users_trfm = df_users_trfm.rename(columns={
    'address.street': 'street',
    'address.city': 'city',
    'address.suite': 'suite',
    'address.zipcode': 'zipcode',
    'company.name': 'company_name'
})
print(df_users_trfm.shape)
print(df_users_trfm.columns)

id                     0
name                   0
username               0
email                  0
phone                  0
website                0
address.street         0
address.suite          0
address.city           0
address.zipcode        0
address.geo.lat        0
address.geo.lng        0
company.name           0
company.catchPhrase    0
company.bs             0
dtype: int64
False    10
Name: count, dtype: int64
(10, 10)
Index(['id', 'name', 'email', 'phone', 'website', 'street', 'suite', 'city',
       'zipcode', 'company_name'],
      dtype='str')


In [9]:
# --- Save transformed layer ---
df_nf_trfm.to_parquet("/home/vinaygautam/etl_learning/data/transformed/netflix_transformed.parquet")
df_users_trfm.to_parquet("/home/vinaygautam/etl_learning/data/transformed/users_transformed.parquet")

In [10]:
# Verify transformed save
df_nf_trfm_check = pd.read_parquet("/home/vinaygautam/etl_learning/data/transformed/netflix_transformed.parquet")
df_users_trfm_check = pd.read_parquet("/home/vinaygautam/etl_learning/data/transformed/users_transformed.parquet")
logger.info(f"Netflix - Raw data: {df_nf.shape} > Transformed data: {df_nf_trfm.shape}")
logger.info(f"Users - Raw data: {df_users.shape} > Transformed data: {df_users_trfm.shape}")

2026-07-18 21:15:31,591 | INFO | Netflix - Raw data: (5283, 11) > Transformed data: (5283, 10)
2026-07-18 21:15:31,594 | INFO | Users - Raw data: (10, 15) > Transformed data: (10, 10)


Day 3: Load

In [11]:
# Connect to (and create) SQLite database
engine = create_engine("sqlite:////home/vinaygautam/etl_learning/etl_learning.db")

In [ ]:
# --- Part A: Full load — Netflix ---
df_nf_trfm.to_sql("netflix", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    result = conn.exec_driver_sql("SELECT COUNT(*) FROM netflix")
    count_nf = result.fetchone()[0]

if df_nf_trfm.shape[0] == count_nf:
    logger.info(f"Netflix load verified: {df_nf_trfm.shape[0]} rows in Dataframe, {count_nf} rows in table. Matched!")
else:
    logger.info(f"Rows mismatch: {df_nf_trfm.shape[0]} -> {count_nf}")

2026-07-18 21:15:32,063 | INFO | Netflix load verified: 5283 rows in Dataframe, 5283 rows in table. Matched!


In [ ]:
# --- Part B: Full load — Users ---
df_users_trfm.to_sql("users", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    result = conn.exec_driver_sql("SELECT COUNT(*) FROM users")
    count_users = result.fetchone()[0]

if df_nf_trfm.shape[0] == count_users:
    logger.info(f"Netflix load verified: {df_users_trfm.shape[0]} rows in Dataframe, {count_users} rows in table. Matched!")
else:
    logger.info(f"Rows mismatch: {df_nf_trfm.shape[0]} -> {count_users}")

2026-07-18 21:15:32,192 | INFO | Transformed DataFrame rows: 10
2026-07-18 21:15:32,198 | INFO | Rows loaded into 'users' table: 10


Day 4: Incremental Upsert simulation

In [14]:
# Read back a real record to use as update template
data = pd.read_sql("SELECT * FROM users", engine)
print(list(data.iloc[0]))

[np.int64(1), 'Leanne Graham', 'Sincere@april.biz', '1-770-736-8031 x56442', 'hildegard.org', 'Kulas Light', 'Apt. 556', 'Gwenborough', '92998-3874', 'Romaguera-Crona']


In [15]:
# t1 = existing record (id=1) with modified email + phone; t2 = new record (id=11)
t1 = [1, 'Leanne Graham', 'Leanne.Graham@april.biz', '1-776-775-5845 x46585', 'hildegard.org', 'Kulas Light', 'Apt. 556', 'Gwenborough', '92998-43874', 'Romaguera-Crona']
t2 = [11, 'John Smith', 'John.Smith@almond.biz', '1-771-734-7865 x68755', 'kontas.org', 'kansas Dark', 'Apt. 7512', 'Miami', '96546-5564', 'Delloite']

# Create incoming_batch dataset from above udpated and new record
incoming_batch = pd.DataFrame([t1, t2], columns=df_users_trfm.columns)

In [16]:
# Split incoming batch: existing (update) vs new (insert)
existing_users = pd.read_sql("SELECT * FROM users", engine)
is_existing = incoming_batch['id'].isin(existing_users['id'])
to_update = incoming_batch[is_existing]
to_insert = incoming_batch[~is_existing]

logger.info(f"To update: {len(to_update)} row(s)")
logger.info(f"To insert: {len(to_insert)} row(s)")

2026-07-18 21:15:32,451 | INFO | To update: 1 row(s)
2026-07-18 21:15:32,456 | INFO | To insert: 1 row(s)


In [17]:
# Insert new records
to_insert.to_sql("users", engine, if_exists="append", index=False)

# Update existing records (parameterized — no string interpolation)
update_stmt = text("UPDATE users SET email = :email, phone = :phone WHERE id = :id")
with engine.connect() as conn:
    for _, row in to_update.iterrows():
        conn.execute(update_stmt, {"email": row["email"], "phone": row["phone"], "id": row["id"]})
    conn.commit()

In [ ]:
# Verify upsert
final = pd.read_sql("SELECT * FROM users", engine)
logger.info(f"Total rows after upsert: {len(final)}")
print(final[final['id'] == 1][['id', 'email', 'phone']])
print(final[final['id'] == 11])

2026-07-18 21:15:32,693 | INFO | Total rows: 11


   id                    email                  phone
0   1  Leanne.Graham@april.biz  1-776-775-5845 x46585
    id        name                  email                  phone     website  \
10  11  John Smith  John.Smith@almond.biz  1-771-734-7865 x68755  kontas.org   

         street      suite   city     zipcode company_name  
10  kansas Dark  Apt. 7512  Miami  96546-5564     Delloite  
